In [ ]:
import string
import pandas as pd
import nltk
import re
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

custom_stops = {
    'subject', 'lines', 'organization', 'writes', 'article',
    'posting', 'host', 'nntp', 'university', 'thanks', 'reply'
}
stop_words.update(custom_stops)

lemma_cache = {}

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def preprocess_fast(text, allowed_pos=['N', 'J']):
    text = re.sub(r'\S*@\S*\s?', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)

    selected_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue

        if any(tag.startswith(pos) for pos in allowed_pos):
            cache_key = (token, tag)
            if cache_key not in lemma_cache:
                lemma_cache[cache_key] = lemmatizer.lemmatize(token, get_wordnet_pos(tag))
            selected_tokens.append(lemma_cache[cache_key])

    return " ".join(selected_tokens)

dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

y_train = train_df['label']
y_test = test_df['label']

pos_tasks = [
    ("Сущ+Прил", ['N', 'J']),
    ("Сущ+Прил+Гл", ['N', 'J', 'V'])
]

lsa_results = []

for label, pos_list in pos_tasks:

    train_txt = [preprocess_fast(t, allowed_pos=pos_list) for t in train_df['text']]
    test_txt = [preprocess_fast(t, allowed_pos=pos_list) for t in test_df['text']]

    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
    X_train_tfidf = tfidf.fit_transform(train_txt)
    X_test_tfidf = tfidf.transform(test_txt)

    for n_comp in [100, 200]:

        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        X_train_lsa = svd.fit_transform(X_train_tfidf)
        X_test_lsa = svd.transform(X_test_tfidf)

        model = HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42
        )

        model.fit(X_train_lsa, y_train)
        y_pred = model.predict(X_test_lsa)

        res = {
            "POS": label,
            "LSA Comp": n_comp,
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3)
        }
        lsa_results.append(res)

df_final = pd.DataFrame(lsa_results)
print("\nИтоговые результаты:")
print(df_final.to_string(index=False))

Repo card metadata block was not found. Setting CardData to empty.



Итоговые результаты:
        POS  LSA Comp  F1 Micro  F1 Macro
   Сущ+Прил       100     0.565     0.551
   Сущ+Прил       200     0.572     0.558
Сущ+Прил+Гл       100     0.562     0.551
Сущ+Прил+Гл       200     0.575     0.563


In [ ]:
import string
import pandas as pd
import nltk
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score

nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
lemma_cache = {}

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def preprocess_fast(text, allowed_pos=['N', 'J']):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)

    selected_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue

        if any(tag.startswith(pos) for pos in allowed_pos):
            #кэширование лемматизации для ускорения
            cache_key = (token, tag)
            if cache_key not in lemma_cache:
                lemma_cache[cache_key] = lemmatizer.lemmatize(token, get_wordnet_pos(tag))
            selected_tokens.append(lemma_cache[cache_key])

    return " ".join(selected_tokens)

dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

y_train = train_df['label']
y_test = test_df['label']

pos_tasks = [
    ("Сущ+Прил", ['N', 'J']),
    ("Сущ+Прил+Гл", ['N', 'J', 'V'])
]

lsa_results = []

for label, pos_list in pos_tasks:

    #предобработка
    train_txt = [preprocess_fast(t, allowed_pos=pos_list) for t in train_df['text']]
    test_txt = [preprocess_fast(t, allowed_pos=pos_list) for t in test_df['text']]

    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
    X_train_tfidf = tfidf.fit_transform(train_txt)
    X_test_tfidf = tfidf.transform(test_txt)

    for n_comp in [100, 200]:

        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        X_train_lsa = svd.fit_transform(X_train_tfidf)
        X_test_lsa = svd.transform(X_test_tfidf)

        model = HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42
        )

        model.fit(X_train_lsa, y_train)
        y_pred = model.predict(X_test_lsa)

        res = {
            "POS": label,
            "LSA Comp": n_comp,
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3)
        }
        lsa_results.append(res)

df_final = pd.DataFrame(lsa_results)
print(df_final.to_string(index=False))

Repo card metadata block was not found. Setting CardData to empty.


        POS  LSA Comp  F1 Micro  F1 Macro
   Сущ+Прил       100     0.557     0.544
   Сущ+Прил       200     0.573     0.560
Сущ+Прил+Гл       100     0.557     0.545
Сущ+Прил+Гл       200     0.571     0.559


In [ ]:
import pandas as pd
import string
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

y_train = train_df['label']
y_test = test_df['label']

# 2. Быстрая предобработка (Basic)
def preprocess_basic(text):
    text = str(text).lower()
    # Удаление пунктуации
    text = text.translate(str.maketrans('', '', string.punctuation))
    return " ".join(text.split())

texts_train_prepared = [preprocess_basic(t) for t in train_df['text']]
texts_test_prepared = [preprocess_basic(t) for t in test_df['text']]

lsa_tuning = []

# Список компонент для проверки
# Примечание: n_components не может быть больше количества признаков в TF-IDF
n_topics_list = [50, 100, 200, 500]

# 3. Эксперимент
for n_topics in n_topics_list:

    # Создаем Pipeline
    lsa_pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1, 1))),
        ('lsa', TruncatedSVD(n_components=n_topics, random_state=42)),
        ('gbm', HistGradientBoostingClassifier(max_iter=100, random_state=42))
    ])

    # Обучение
    lsa_pipe.fit(texts_train_prepared, y_train)

    # Предсказание
    y_pred = lsa_pipe.predict(texts_test_prepared)

    # Метрики
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    lsa_tuning.append({
        "Model": "HistGBM + LSA",
        "Topics (n_comp)": n_topics,
        "F1 Micro": round(f1_micro, 3),
        "F1 Macro": round(f1_macro, 3)
    })

# Результаты
df_lsa = pd.DataFrame(lsa_tuning)
print("\nРезультаты LSA эксперимента на 20_newsgroups:")
print(df_lsa.to_string(index=False))

Repo card metadata block was not found. Setting CardData to empty.



Результаты LSA эксперимента на 20_newsgroups:
        Model  Topics (n_comp)  F1 Micro  F1 Macro
HistGBM + LSA               50     0.537     0.526
HistGBM + LSA              100     0.559     0.546
HistGBM + LSA              200     0.580     0.567
HistGBM + LSA              500     0.581     0.568


In [ ]:
import numpy as np
import pandas as pd
import string
import warnings
from datasets import load_dataset

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

# Полное подавление системных уведомлений
warnings.filterwarnings("ignore")

# 1. Загрузка данных
dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

def preprocess(text):
    return str(text).lower().translate(str.maketrans('', '', string.punctuation))

X_train_raw = [preprocess(t) for t in train_df['text']]
X_test_raw = [preprocess(t) for t in test_df['text']]
y_train = train_df['label']
y_test = test_df['label']

# 2. Векторизация
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

# 3. Список моделей и их параметров
# Исправлена ошибка в кавычках для HistGBM
models_data = [
    {
        "name": "Decision Tree",
        "model": DecisionTreeClassifier(random_state=42),
        "params": {'max_depth': [10, 50], 'min_samples_leaf': [1, 5]}
    },
    {
        "name": "Random Forest",
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {'n_estimators': [100], 'max_depth': [20, None]}
    },
    {
        "name": "HistGBM (LSA)",
        "model": Pipeline([
            ('lsa', TruncatedSVD(n_components=100, random_state=42)),
            ('gbm', HistGradientBoostingClassifier(random_state=42))
        ]),
        "params": {
            'gbm__max_iter': [50, 100],
            'gbm__learning_rate': [0.1]
        }
    },
    {
        "name": "AdaBoost",
        "model": AdaBoostClassifier(random_state=42),
        "params": {'n_estimators': [50], 'learning_rate': [0.1, 1.0]}
    }
]

final_results = []

# 4. Цикл автоматического поиска
for m in models_data:
    search = HalvingGridSearchCV(
        m["model"],
        m["params"],
        factor=3,
        cv=2,
        scoring='f1_micro',
        n_jobs=-1,
        verbose=0
    )

    search.fit(X_train, y_train)

    # Тест на отложенной выборке
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    f1_res = round(f1_score(y_test, y_pred, average='micro'), 3)

    final_results.append({
        "Model": m["name"],
        "Best Params": str(search.best_params_),
        "F1 Micro (Test)": f1_res
    })

# 5. Итоговая таблица
df_results = pd.DataFrame(final_results)
print(df_results.to_string(index=False))

Repo card metadata block was not found. Setting CardData to empty.


        Model                                       Best Params  F1 Micro (Test)
Decision Tree          {'max_depth': 50, 'min_samples_leaf': 1}            0.332
Random Forest          {'max_depth': None, 'n_estimators': 100}            0.573
HistGBM (LSA) {'gbm__learning_rate': 0.1, 'gbm__max_iter': 100}            0.540
     AdaBoost        {'learning_rate': 1.0, 'n_estimators': 50}            0.261


In [ ]:
import pandas as pd
import numpy as np
import string
from datasets import load_dataset
from sklearn.model_selection import cross_validate, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset['train'])

def preprocess_basic(text):
    return str(text).lower().translate(str.maketrans('', '', string.punctuation))

texts_cv = [preprocess_basic(t) for t in train_df['text']]
y_cv = np.array(train_df['label'])

tfidf_cv = TfidfVectorizer(max_features=5000, stop_words='english')
X_cv = tfidf_cv.fit_transform(texts_cv)

models_to_compare = [
    ("Decision Tree", DecisionTreeClassifier(max_depth=50, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=50, random_state=42, n_jobs=-1))
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_cv_results = []


for name, model in models_to_compare:

    cv_output = cross_validate(
        model,
        X_cv,
        y_cv,
        cv=kf,
        scoring=['f1_micro', 'f1_macro'],
        return_train_score=False,
        n_jobs=-1
    )

    all_cv_results.append({
        "Model": name,
        "F1 Micro (Mean)": round(cv_output['test_f1_micro'].mean(), 3),
        "F1 Micro (Std)": round(cv_output['test_f1_micro'].std(), 4),
        "F1 Macro (Mean)": round(cv_output['test_f1_macro'].mean(), 3)
    })

print("\nСравнение моделей (CV=5) на датасете 20_newsgroups")
df_comparison = pd.DataFrame(all_cv_results)
print(df_comparison.to_string(index=False))

Repo card metadata block was not found. Setting CardData to empty.



Сравнение моделей (CV=5) на датасете 20_newsgroups
        Model  F1 Micro (Mean)  F1 Micro (Std)  F1 Macro (Mean)
Decision Tree            0.370          0.0049            0.418
Random Forest            0.588          0.0084            0.591
